# Homework 4 - Evaluation Metrics for Classification


In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, KFold

df = pd.read_csv("course_lead_scoring_2026.csv")
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data preparation



In [2]:
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = df.drop(columns=['converted']).select_dtypes(include='number').columns.tolist()

df_clean = df.copy()
df_clean[cat_cols] = df_clean[cat_cols].fillna('NA')
df_clean[num_cols] = df_clean[num_cols].fillna(0.0)

df_full_train, df_test = train_test_split(df_clean, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train.pop('converted')
y_val = df_val.pop('converted')

print(len(df_train), len(df_val), len(df_test))

3000 1000 1000


## Q1. ROC AUC feature importance



In [3]:
aucs = {}
for col in num_cols:
    score = roc_auc_score(y_train, df_train[col])
    if score < 0.5:
        score = roc_auc_score(y_train, -df_train[col])
    aucs[col] = score

candidates = ['lead_score', 'number_of_courses_viewed', 'interaction_count', 'annual_income']
for c in candidates:
    print(f'{c:>25}: {aucs[c]:.4f}')
print('-> Mayor AUC:', max(candidates, key=aucs.get))

               lead_score: 0.7882
 number_of_courses_viewed: 0.7230
        interaction_count: 0.7649
            annual_income: 0.6082
-> Mayor AUC: lead_score


## Helper: one-hot encoding + logistic regression

In [4]:
def train_model(X_tr, y_tr, X_va, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_tr_vec = dv.fit_transform(X_tr.to_dict(orient='records'))
    X_va_vec = dv.transform(X_va.to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000)
    model.fit(X_tr_vec, y_tr)
    return model, X_va_vec

## Q2. AUC of the model on validation



In [5]:
model, X_val_vec = train_model(df_train, y_train, df_val)
probs = model.predict_proba(X_val_vec)[:, 1]
auc_val = roc_auc_score(y_val, probs)
print('-> AUC:', round(auc_val, 3))

-> AUC: 0.732


## Q3. Precision and recall intersection



In [6]:
thresholds = np.arange(0.0, 1.0, 0.01)
best = None
for t in thresholds:
    pred = (probs >= t).astype(int)
    p = precision_score(y_val, pred, zero_division=0)
    r = recall_score(y_val, pred, zero_division=0)
    if p == 0 and r == 0:
        continue
    if best is None or abs(p - r) < best[0]:
        best = (abs(p - r), t)
print('-> Intersection threshold:', round(best[1], 2))

-> Intersection threshold: 0.63


## Q4. F1 score



In [7]:
f1s = [f1_score(y_val, (probs >= t).astype(int), zero_division=0) for t in thresholds]
best_idx = int(np.argmax(f1s))
print('-> Threshold F1 max:', round(thresholds[best_idx], 2))

-> Threshold F1 max: 0.41


## Q5. 5-Fold CV



In [8]:
cv = KFold(n_splits=5, shuffle=True, random_state=1)
scores = []
for tr_idx, va_idx in cv.split(df_full_train):
    tr = df_full_train.iloc[tr_idx].copy()
    va = df_full_train.iloc[va_idx].copy()
    tr_y = tr.pop('converted')
    va_y = va.pop('converted')
    m, vec = train_model(tr, tr_y, va)
    scores.append(roc_auc_score(va_y, m.predict_proba(vec)[:, 1]))

print('Scores:', [round(s, 3) for s in scores])
print('-> std:', round(float(np.std(scores)), 3))

Scores: [0.731, 0.73, 0.736, 0.741, 0.721]
-> std: 0.007


## Q6. Hyperparameter tuning



In [9]:
for C in [0.000001, 0.001, 1]:
    means, stds = [], []
    for tr_idx, va_idx in cv.split(df_full_train):
        tr = df_full_train.iloc[tr_idx].copy()
        va = df_full_train.iloc[va_idx].copy()
        tr_y = tr.pop('converted')
        va_y = va.pop('converted')
        m, vec = train_model(tr, tr_y, va, C=C)
        means.append(roc_auc_score(va_y, m.predict_proba(vec)[:, 1]))
    print(f'C = {C:<10} mean = {round(np.mean(means), 3)}  std = {round(np.std(means), 3)}')
print('\n-> Mejor C: 0.001')

C = 1e-06      mean = 0.617  std = 0.019


C = 0.001      mean = 0.749  std = 0.01


C = 1          mean = 0.732  std = 0.007

-> Mejor C: 0.001
